In [ ]:
from neo4j import GraphDatabase

# Function to connect to Neo4j
def connect_to_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    return driver

# Connect to a Neo4j instance which enables Neo4j GDS, e. g. a local database
# adjust credentials according to your Neo4j instance
uri = "bolt://localhost:7689" # alternative: bolt://127.0.0.1:7687
user = "neo4j"
password = "password"
NEO4J_DB = "neo4j"

driver = connect_to_neo4j(uri, user, password)


In [2]:
# getting started with Neo4j Graph Data Science

from graphdatascience import GraphDataScience
gds = GraphDataScience(uri, auth=(user, password), database=NEO4J_DB)

# Check the installed GDS version on the server

print(gds.version())
assert gds.version() 

/home/ssc/projects/MI4People_Care_4_Rare_Five1/careforrare/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.6.8


In [ ]:
## project training graph
## here, Biological_sample for training and test graph were not randomly sampled, but selected based on subjectid
## subjectid's may not be up-to-date anymore in newer dump-files for local databases
## -> please adjust the query accordingly or use the sampling approach in GDS_link_prediction_composite.py


G_train_exists = gds.run_cypher("""CALL gds.graph.exists("train_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_train_exists.iloc[0,0]==True:
    gds.graph.drop("train_graph")


#create graph projection
G_train, result = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "10") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "40") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "41") OR
           source:Phenotype OR 
           source:Protein OR 
           source:Disease                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PARENT|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE]->(target)
            WHERE target:Phenotype OR                                                                      
            target:Gene OR
            target:Protein OR
            target:Disease                               
    RETURN gds.graph.project(
    'train_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    //sourceNodeProperties: source { .subjectid, .id},                                           
    targetNodeLabels: labels(target),
    //targetNodeProperties: target { .id},                                           
    relationshipType: type(r),
    relationshipProperties: r { score: coalesce(r.score, 0.0) }
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")

assert G_train.node_count() == result["nodeCount"]

In [3]:
## project test graph

G_test_exists = gds.run_cypher("""CALL gds.graph.exists("test_graph") YIELD exists""")

#pipe_cli_exists = gds.run_cypher("""CALL gds.pipeline.exists('pipe_cli') YIELD exists""")

if G_test_exists.iloc[0,0]==True:
    gds.graph.drop("test_graph")

#create graph projection
G_test, result_test = gds.graph.cypher.project("""MATCH (source)
    WHERE (source:Biological_sample AND source.subjectid STARTS WITH "42") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "43") OR
          (source:Biological_sample AND source.subjectid STARTS WITH "44") OR
           source:Phenotype OR
           source:Protein OR
           source:Disease                                                                                                                                  
    OPTIONAL MATCH (source)-[r:HAS_PHENOTYPE|HAS_DAMAGE|HAS_PROTEIN|COMPILED_INTERACTS_WITH|HAS_DISEASE|IS_BIOMARKER_OF_DISEASE]->(target)
            WHERE target:Gene OR target:Protein OR target:Phenotype OR
            (source:Biological_sample AND target:Disease AND target.id STARTS WITH "DOID:4")
    RETURN gds.graph.project(
    'test_graph',
    source,
    target,
    {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: type(r),
    relationshipProperties: r { score: coalesce(r.score, 0)}
    },
    { undirectedRelationshipTypes: ['HAS_DISEASE']}                                    
    )""")
    
assert G_test.node_count() == result_test["nodeCount"]

In [21]:
#gds.graph.drop("train_graph")

gds.graph.list()

,degreeDistribution,graphName,database,databaseLocation,memoryUsage,sizeInBytes,nodeCount,relationshipCount,configuration,density,creationTime,modificationTime,schema,schemaWithOrientation
0,"{'min': 0, 'max': 1970, 'p90': 0, 'p999': 504,...",test_graph,neo4j,local,182 MiB,191321656,255581,1959848,"{'readConcurrency': 4, 'undirectedRelationship...",0.000030,2024-11-14T14:10:51.579569000+00:00,2024-11-14T14:11:24.893909000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."
1,"{'min': 0, 'max': 1970, 'p90': 2, 'p999': 504,...",train_graph,neo4j,local,300 MiB,314827072,255795,1999483,"{'readConcurrency': 4, 'undirectedRelationship...",0.000031,2024-11-14T14:10:12.152862000+00:00,2024-11-14T14:10:50.832975000+00:00,"{'graphProperties': {}, 'nodes': {'Phenotype':...","{'graphProperties': {}, 'nodes': {'Phenotype':..."


In [ ]:
# stream HAS_DISEASE relationships from test graph -> which patients have which diseases? does this have an impact on the prediction?
gds.run_cypher("""CALL gds.graph.relationships.stream(
                        'test_graph',
                       ['HAS_DISEASE']
                        )
                        YIELD sourceNodeId, targetNodeId, relationshipType 
                        RETURN gds.util.asNode(sourceNodeId).subjectid as patient_id, gds.util.asNode(targetNodeId).id as disease_id
                        ORDER BY patient_id""")

,patient_id,disease_id
0,42199,DOID:4372
1,42430,DOID:4372
2,43870,DOID:4372
3,None,None
4,None,None
5,None,None


##### approximate inductive link prediction using GraphSAGE for node embedding (Cypher)

In [22]:
## GraphSAGE requires NodeProperties for node embedding -> run Louvain and Node2Vec analogous to FastRP

In [23]:
#  Louvain algorithm for train_graph

gds.run_cypher("""CALL gds.louvain.mutate("train_graph", 
               {maxIterations: 10, 
               relationshipWeightProperty: 'score', 
               mutateProperty: "community"}) YIELD nodePropertiesWritten""")



,nodePropertiesWritten
0,255795


In [24]:
#  Louvain algorithm for test_graph

gds.run_cypher("""CALL gds.louvain.mutate("test_graph", 
               {maxIterations: 10, 
               relationshipWeightProperty: 'score', 
               mutateProperty: "community"}) YIELD nodePropertiesWritten""")



,nodePropertiesWritten
0,255581


In [25]:
# degree centrality

#  Degree centrality for train_graph
gds.run_cypher("""CALL gds.degree.mutate('train_graph', 
               { mutateProperty: 'degree', 
               relationshipWeightProperty: 'score' 
               }) 
               YIELD centralityDistribution, nodePropertiesWritten""")

#  Degree centrality for test_graph
gds.run_cypher("""CALL gds.degree.mutate('test_graph', 
               { mutateProperty: 'degree', 
               relationshipWeightProperty: 'score' 
               }) 
               YIELD centralityDistribution, nodePropertiesWritten""")

,centralityDistribution,nodePropertiesWritten
0,"{'min': 0.0, 'max': 1354.6562480926514, 'p90':...",255581


In [26]:
# similarity algorithms
## do not include scores to only calculate similarity based on structure; naturally, isolated nodes are not considered for similarity calculation

# Node Similarity train_graph
#gds.run_cypher("""CALL gds.nodeSimilarity.mutate('train_graph', 
#               {
#               mutateRelationshipType: 'SIMILAR_TO',
#               mutateProperty: 'score', 
#               //relationshipWeightProperty: 'score', 
#               topK: 1
#               }) 
#               YIELD nodesCompared, relationshipsWritten
#              """)

In [27]:
# similarity algorithms

# Node Similarity test_graph
#gds.run_cypher("""CALL gds.nodeSimilarity.mutate('test_graph', 
#               {
#               mutateRelationshipType: 'SIMILAR_TO',
#               mutateProperty: 'score',  
#               //relationshipWeightProperty: 'score', 
#               topK: 1}) 
#               YIELD nodesCompared, relationshipsWritten""")

In [ ]:
# FastRP node embeddings for train_graph

gds.run_cypher("""CALL gds.fastRP.mutate("train_graph",
    {mutateProperty: 'fastRP',
    featureProperties: ['community', 'degree'],                     
    relationshipWeightProperty: 'score',
    embeddingDimension: 256,
    propertyRatio: 1.0,                      
    randomSeed: 42 }) YIELD nodePropertiesWritten
    """)

,nodePropertiesWritten
0,255795


In [29]:
# FastRP node embeddings for test_graph

gds.run_cypher("""CALL gds.fastRP.mutate("test_graph",
    {mutateProperty: 'fastRP',
    featureProperties: ['community', 'degree'],           
    relationshipWeightProperty: 'score',
    embeddingDimension: 256,
    propertyRatio: 1.0,           
    randomSeed: 42}) YIELD nodePropertiesWritten
    """)

,nodePropertiesWritten
0,255581


In [ ]:
## GraphSAGE - train GraphSAGE outside of the pipeline

if gds.run_cypher("""CALL gds.model.exists('graphsage') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.model.drop('graphsage')""")

#if gds.run_cypher("""CALL gds.model.exists('graphsage') YIELD exists""").iloc[0,0]==False:
gds.run_cypher("""CALL gds.beta.graphSage.train('train_graph', 
                   {modelName: 'graphsage',
                   relationshipWeightProperty: 'score',
                   featureProperties: ['fastRP']
                   })""")

## note: did not include relationshipWeightProperty, as only HAS_DAMAGE and HAS_PROTEIN are not NaN -> NaN throws an error
## -> can avoid this error by coalescing the score to 0.0 in the projection
## featureProperties: ['community'] -> only community yields non-informative predictions
## leave-out Louvain community for now; maybe consider another algorithm

,modelInfo,configuration,trainMillis
0,"{'modelName': 'graphsage', 'modelType': 'graph...","{'aggregator': 'MEAN', 'jobId': '794e6c04-146f...",97725


In [31]:
# pipeline configuration - Cypher

if gds.run_cypher("""CALL gds.pipeline.exists('pipe_sage') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.pipeline.drop('pipe_sage')""")



In [32]:

gds.beta.pipeline.linkPrediction.create('pipe_sage')

# add node property
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addNodeProperty('pipe_sage', 'gds.beta.graphSage', {
    modelName: 'graphsage',
    mutateProperty: 'embedding',           
    contextNodeLabels: ['Protein', 'Gene', 'Phenotype'],
    contextRelationshipTypes: ['HAS_PROTEIN', 'HAS_DAMAGE', 'COMPILED_INTERACTS_WITH', 'HAS_PARENT', 'HAS_PHENOTYPE', 'IS_BIOMARKER_OF_DISEASE']
    })""")


,name,nodePropertySteps,featureSteps,splitConfig,autoTuningConfig,parameterSpace
0,pipe_sage,"[{'name': 'gds.beta.graphSage.mutate', 'config...",[],"{'testFraction': 0.1, 'validationFolds': 3, 't...",{'maxTrials': 10},"{'MultilayerPerceptron': [], 'RandomForest': [..."


In [ ]:
if gds.run_cypher("""CALL gds.model.exists('pheno-sage') YIELD exists""").iloc[0,0]==True:
    gds.run_cypher("""CALL gds.model.drop('pheno-sage')""")


# add link features
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addFeature('pipe_sage', 'cosine', {
    nodeProperties: ['embedding']
})""")


#Configuring the relationship split -> what do you need the feature input for?
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.configureSplit('pipe_sage', {
    testFraction: 0.2,
    trainFraction: 0.6,
    validationFolds: 3
    //negativeSamplingRatio: 1000.0           
})""")


# add model candidates
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addLogisticRegression('pipe_sage')""")
gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.addRandomForest('pipe_sage', {numberOfDecisionTrees: 100})""")
gds.run_cypher(""" CALL gds.alpha.pipeline.linkPrediction.addMLP('pipe_sage', {hiddenLayerSizes: [64, 32], penalty: 0.01, patience: 2})""")

# memory estimation
#gds.run_cypher(""" CALL gds.beta.pipeline.linkPrediction.train.estimate('train_graph2', {
#               pipeline: 'pipe_sage',
#               modelName: 'pheno-sage',
#               targetRelationshipType: 'HAS_PHENOTYPE'
#               })""")


# training -> adjust source & target node
gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.train('train_graph', {
  pipeline: 'pipe_sage',
  modelName: 'pheno-sage',
  metrics: ['AUCPR', 'OUT_OF_BAG_ERROR'],
  //negativeClassWeight: 0.001,
  sourceNodeLabel: 'Biological_sample',
  targetNodeLabel: 'Disease',             
  targetRelationshipType: 'HAS_DISEASE',
  randomSeed: 42
}) YIELD modelInfo, modelSelectionStats
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.AUCPR.train.avg AS avgTrainScore,
  modelInfo.metrics.AUCPR.outerTrain AS outerTrainScore,
  modelInfo.metrics.AUCPR.test AS testScore,
  [cand IN modelSelectionStats.modelCandidates | cand.metrics.AUCPR.validation.avg] AS validationScores""")



,winningModel,avgTrainScore,outerTrainScore,testScore,validationScores
0,"{'minEpochs': 1, 'maxEpochs': 100, 'focusWeigh...",0.891636,0.891516,0.762562,"[0.8925426059512153, 0.8925426059512153, 0.892..."


In [45]:
model = gds.run_cypher("""CALL gds.model.list("pheno-sage")""")

model

,modelName,modelType,modelInfo,creationTime,trainConfig,graphSchema,loaded,stored,published
0,pheno-sage,LinkPrediction,{'metrics': {'AUCPR': {'test': 0.7625622346138...,2024-11-14T14:18:10.474454000+00:00,"{'randomSeed': 42, 'targetRelationshipType': '...","{'graphProperties': {}, 'nodes': {'Disease': {...",True,False,False


In [36]:
predict_sage = gds.run_cypher("""CALL gds.beta.pipeline.linkPrediction.predict.stream('test_graph', {
  modelName: 'pheno-sage',
  topN: 100,
  sampleRate: 1.0,
  threshold: 0.1
})
 YIELD node1, node2, probability
 //WHERE gds.util.asNode(node1).id STARTS WITH "HP:0040263"
 //WHERE NOT gds.util.asNode(node2).subjectid STARTS WITH "4"
 //RETURN DISTINCT gds.util.asNode(node1).id as node1
 RETURN gds.util.asNode(node1).id AS disease_id, gds.util.asNode(node2).subjectid AS patient_id, probability
 //RETURN DISTINCT gds.util.asNode(node2).subjectid AS sample, COLLECT(DISTINCT gds.util.asNode(node1).id) AS disease, COUNT(DISTINCT gds.util.asNode(node1).id) AS count                             
 ORDER BY gds.util.asNode(node2).subjectid""")

predict_sage

,disease_id,patient_id,probability
0,DOID:4372,42033,1.00
1,DOID:4372,42066,1.00
2,DOID:14449,42066,0.46
3,DOID:0110462,42066,0.46
4,DOID:0060694,42066,0.46
...,...,...,...
95,DOID:0110462,44228,0.46
96,DOID:0060694,44228,0.46
97,DOID:0110463,44228,0.46
98,DOID:14451,44228,0.46


In [37]:
predict_sage

,disease_id,patient_id,probability
0,DOID:4372,42033,1.00
1,DOID:4372,42066,1.00
2,DOID:14449,42066,0.46
3,DOID:0110462,42066,0.46
4,DOID:0060694,42066,0.46
...,...,...,...
95,DOID:0110462,44228,0.46
96,DOID:0060694,44228,0.46
97,DOID:0110463,44228,0.46
98,DOID:14451,44228,0.46


In [ ]:
# compare predictions to actual data

ctrl = gds.run_cypher("""MATCH (bs:Biological_sample)
               WHERE bs.subjectid STARTS WITH "42" OR bs.subjectid STARTS WITH "43" OR bs.subjectid STARTS WITH "44"
               MATCH (bs)-[:HAS_DISEASE]->(d:Disease)
               RETURN d.id as disease_id, bs.subjectid as patient_id
               ORDER BY patient_id""")
ctrl

,disease_id,patient_id
0,DOID:10030,42075
1,DOID:6376,42075
2,DOID:10030,42135
3,DOID:4372,42199
4,DOID:112,42281
5,DOID:1680,42281
6,DOID:10030,42281
7,DOID:5295,42281
8,DOID:10230,42292
9,DOID:11963,42292


In [39]:
import pandas as pd

# Step 1: Merge DataFrames on both disease_id and patient_id
correct_predictions = pd.merge(predict_sage, ctrl, on=['disease_id', 'patient_id'])

# Step 2: Identify false positives (predictions not in actual data)
false_positives = pd.merge(predict_sage, correct_predictions, how='left', indicator=True)
false_positives = false_positives[false_positives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Step 3: Identify false negatives (actual data not in predictions)
false_negatives = pd.merge(ctrl, correct_predictions, how='left', indicator=True)
false_negatives = false_negatives[false_negatives['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display results
print("Correct Predictions:")
print(correct_predictions)

print("\nFalse Positives (Predicted but not actual):")
print(false_positives)

print("\nFalse Negatives (Actual but not predicted):")
print(false_negatives)

Correct Predictions:
Empty DataFrame
Columns: [disease_id, patient_id, probability]
Index: []

False Positives (Predicted but not actual):
      disease_id patient_id  probability
0      DOID:4372      42033         1.00
1      DOID:4372      42066         1.00
2     DOID:14449      42066         0.46
3   DOID:0110462      42066         0.46
4   DOID:0060694      42066         0.46
..           ...        ...          ...
95  DOID:0110462      44228         0.46
96  DOID:0060694      44228         0.46
97  DOID:0110463      44228         0.46
98    DOID:14451      44228         0.46
99     DOID:6522      44228         0.46

[100 rows x 3 columns]

False Negatives (Actual but not predicted):
      disease_id patient_id  probability
0     DOID:10030      42075          NaN
1      DOID:6376      42075          NaN
2     DOID:10030      42135          NaN
3      DOID:4372      42199          NaN
4       DOID:112      42281          NaN
5      DOID:1680      42281          NaN
6     DOID:10

In [40]:
# Step 1: Group by patient_id and collect sets of disease_id
predicted_groups = predict_sage.groupby('patient_id')['disease_id'].apply(set).reset_index()
actual_groups = ctrl.groupby('patient_id')['disease_id'].apply(set).reset_index()

# Step 2: Merge on patient_id to align predicted and actual disease sets per patient
merged_df = pd.merge(predicted_groups, actual_groups, on='patient_id', how='outer', suffixes=('_predicted', '_actual'))

# Step 3: Fill NaN values with empty sets
merged_df['disease_id_predicted'] = merged_df['disease_id_predicted'].apply(lambda x: x if isinstance(x, set) else set())
merged_df['disease_id_actual'] = merged_df['disease_id_actual'].apply(lambda x: x if isinstance(x, set) else set())

# Step 4: Check for any overlap in disease sets for each patient
merged_df['has_overlap'] = merged_df.apply(lambda row: bool(row['disease_id_predicted'] & row['disease_id_actual']), axis=1)

# Display results
print("Disease Prediction Comparison:")
print(merged_df[['patient_id', 'disease_id_predicted', 'disease_id_actual', 'has_overlap']])

Disease Prediction Comparison:
   patient_id                               disease_id_predicted  \
0       42033                                        {DOID:4372}   
1       42066  {DOID:0110463, DOID:14449, DOID:4372, DOID:144...   
2       42075  {DOID:6512, DOID:0110463, DOID:14449, DOID:651...   
3       42135                                        {DOID:4372}   
4       42199  {DOID:6512, DOID:0110463, DOID:14449, DOID:651...   
5       42231                                        {DOID:4372}   
6       42275                                        {DOID:4372}   
7       42281                                        {DOID:4372}   
8       42292                                        {DOID:4372}   
9       42302                                        {DOID:4372}   
10      42321                                        {DOID:4372}   
11      42346                                        {DOID:4372}   
12      42367  {DOID:14451, DOID:6512, DOID:0110463, DOID:652...   
13      42412    

In [41]:
# Step 1: Extract unique disease IDs as sets
predicted_diseases = set(predict_sage['disease_id'].unique())
actual_diseases = set(ctrl['disease_id'].unique())

# Step 2: Check for overlap
overlap = predicted_diseases & actual_diseases  # Intersection of both sets

# Results
if overlap:
    print("Overlap exists. The following disease IDs are present in both predicted and actual data:")
    print(overlap)
else:
    print("No overlap found between predicted and actual disease IDs.")

Overlap exists. The following disease IDs are present in both predicted and actual data:
{'DOID:4372'}
